In [3]:
import numpy as np
import math  as math
import time  as tm
import pickle
import random
import matplotlib.pyplot as plt

def generate_linear(x, d, n):

    A = np.random.normal(0, 1, (n, d))

    E = np.random.normal(0, 1, n)

    B = np.dot(A, x) + E
    return A, B

def generate_logistic(x, d, n):
    A = np.random.normal(0, 1, (n, d))

    P = 1 / (1 + np.exp(-np.dot(A, x)))

    B = np.random.binomial(n=1, p=P)
    return A, B


def grad_linear(x, a, b):
    return a*(np.dot(a, x) - b) 

def grad_logistic(x, a, b):
    hat_b = 1/(1 + np.exp(-np.dot(a, x)))
    return a*(hat_b - b)

class SGD:
    def __init__(self, d, alpha = 0.505, eta = 0.1, grad=grad_linear):
        self.sgd = np.zeros(d)
        self.nn = 0
        self.alpha = alpha
        self.eta = eta
        self.grad = grad
        

        

    def update(self, a, b):
        self.nn = self.nn + 1
        self.sgd = self.sgd - self.eta*self.nn**(-self.alpha)*self.grad(self.sgd, a, b)




def inferenceWASGD(x0, d, alpha, eta, grad, model, n_rep, n_sample, burnin=0, K=6, qt=2.570581836612853, eta_pda=3): 
  
    
    methods = ['ow', 'simple', 'tail', 'pda']
    

    metrics = {m: {'cov': np.zeros(n_rep), 'len': np.zeros(n_rep), 'est': np.zeros(n_rep)} for m in methods}

    for rep in range(n_rep):
        rep_start_time = tm.time()  
        

        estimates = {m: np.zeros((K, d)) for m in methods}
        

        for k in range(K):
            sgd = SGD(d, alpha, eta, grad)
            data_a, data_b = model(x0, d, n_sample)
            

            ow_avg = np.zeros(d)
            simple_avg = np.zeros(d)
            sum_tail = np.zeros(d)
            pda_avg = np.zeros(d)
            
            for step in range(n_sample):
                x_old = sgd.sgd.copy()
                

                sgd.update(data_a[step], data_b[step])
                
                x_new = sgd.sgd.copy()
                i = sgd.nn  
                
          
                if i > burnin:

                    t = i - burnin  
                    
                    if t == 1:
                        ow_avg = x_new.copy()
                    else:
                        ow_avg = ((t - 1) / t) * ow_avg + (1 - t**alpha) * (x_old / t) + (t**(alpha - 1)) * x_new

                    if t == 1:
                        simple_avg = x_new.copy()
                    else:
                        simple_avg = ((t - 1) / t) * simple_avg + (1 / t) * x_new
                        

                    if t == 1:
                        pda_avg = x_new.copy()
                    else:
                        pda_avg = (1 - (eta_pda + 1) / (t + eta_pda)) * pda_avg + ((eta_pda + 1) / (t + eta_pda)) * x_new


                if i > n_sample // 2:
                    sum_tail += x_new


            estimates['ow'][k] = ow_avg.copy()
            estimates['simple'][k] = simple_avg.copy()

            estimates['tail'][k] = sum_tail / (n_sample - n_sample // 2)
            estimates['pda'][k] = pda_avg.copy()


        for m in methods:
            est_k = estimates[m]
            

            asgd_mean = np.mean(est_k, axis=0)
            asgd_std = np.std(est_k, axis=0, ddof=1)
            

            margin = qt * (asgd_std / np.sqrt(K))
            

            cov_mask = (x0 >= asgd_mean - margin) & (x0 <= asgd_mean + margin)

            metrics[m]['cov'][rep] = np.sum(cov_mask) / d
            metrics[m]['len'][rep] = np.mean(2 * margin)
            metrics[m]['est'][rep] = np.mean((x0 - asgd_mean)**2)

        rep_end_time = tm.time()  
        print(f"Replication {rep + 1}/{n_rep} finished in {rep_end_time - rep_start_time:.4f} seconds.")


    final_results = {}
    for m in methods:
        final_cov = np.mean(metrics[m]['cov'])
        len_mean = np.mean(metrics[m]['len'])
        len_std = np.std(metrics[m]['len'])
        final_est = np.mean(metrics[m]['est'])
        
        final_results[m] = [final_cov, len_mean, len_std, final_est]
        
    return final_results

In [ ]:
d = 5
x0 = np.array([_ for _ in range(d)])/(d-1)
n_sample = 500
alpha = 0.505
eta = 1
n_rep = 500
grad = grad_linear
model = generate_linear

np.random.seed(2025)
linearWASGD = inferenceWASGD(x0, d, alpha, eta, grad, model, n_rep, n_sample, burnin = 0, K = 6, qt = 2.570581836612853, eta_pda=3)

In [ ]:
np.random.seed(2025)
grad = grad_logistic
model = generate_logistic
logisticWASGD = inferenceWASGD(x0, d, alpha, eta, grad, model, n_rep, n_sample = 500, burnin = 0, K = 6, qt = 2.570581836612853, eta_pda=3)